In [4]:
#testing Data analysis model on HSI.xlsx for Hang Seng Index, the excel file has many missing data values for market sentiment and is a classic time series model.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

'''
    I am using Time window based imputation using Absolute Median Deviation to assume regime of imputation, through market stability/volatility assumption based on Up votes and Down votes to discern sentiment
    and local market dynamics, the ultimate goal being to analyse whether the market sentiment follows through with market performance or is contrarion or non-aligned with market performance. Time window
    here is 3 days, for a local snapshot of market dynamics, which follows from the calculation of time-series median values whereby the dataset of readings of upvotes within the time window are ordered
    not by magnitude but by position in time series, allowing for great simplification methods of MAD = MAD_t = median{ |x_t - x_(t-1)|, |x_(t-1) - x_(t-2)|, |x_(t-2) - x(t-3)| } for window of 3 days
    which arises from M_t ≈ x_{t-1}, where M_t is the aforementioned time-series median values, which is substituted into MAD_t = median{ |x_t - M_t|, |x_{t-1} - M_{t-1}|, |x_{t-2} - M_{t-2}| }. But for
    the case of rigourousness this code sticks to the unsubstituted MAD_t formula.
 '''

def HSI_Analyze(window=3, MAD_threshold=5):                         #there used to be another parameter here to pass the HSI.xlsx file into this function which allows us to pass any datafile but since this follows just 1 .xlsx file's analysis i decided to remove it
    HSI_dataset = pd.read_excel("HSI.xlsx", parse_dates=['date'])
    window_opens = []                                               # These 3 arrays, window_open, window_close, and window_return are used to figure out each window's start and end inside relevant to a later loop so the final comparison between vote and market closes is computed per window instead of using close.pct_change() between intraday market closes which represents the dynamics relevant to a smaller window than ours: the row-to-row change across the full series.
    window_closes = []
    window_returns = []
                                                                    # Adopting header data from the excel sheet and relabelling for more analysis friendly names (spaces cause a few issues)
    HSI_dataset.columns = [cols.strip().lower().replace(' ', '_') for cols in HSI_dataset.columns]
    for col_count in ['up_votes', 'down_votes', 'close']:
        HSI_dataset[col_count] = pd.to_numeric(HSI_dataset[col_count].astype(str).str.replace(',', ''), errors='coerce')

                                                                     # I am going to keep original data in a new field within the Dataframe, so that when we finally output a CSV based on the dataframe we can compare by eye
    HSI_dataset['up_orig'] = HSI_dataset['up_votes'].copy()
    HSI_dataset['down_orig'] = HSI_dataset['down_votes'].copy()
    
                                                                   # It is better to store all our MAD values so that we will have some data to double check in a CSV and parse through excel for further analysis. We will need a dynamic array to store this for best practices, in case we extend all data points in the original .xlsx file, which is why np.nan comes in as the 2nd parameter in np.full to create a large array
    MAD_up = np.full(len(HSI_dataset), np.nan)                     #stores the MAD for the upvotes
    MAD_down = np.full(len(HSI_dataset), np.nan)
    HSI_dataset['close_percent_change_window'] = np.nan
    
    
    for i in range(window-1, len(HSI_dataset)):
        window_up = HSI_dataset['up_votes'].iloc[(i-window)+1 : i+1]                   #The window is chosen as [x_(t-2), x_(t-1), x_t] for both upvotes and downvotes where x is a dummy variable here, and can represent u or d, where u = upvotes set and d = downvotes set
        window_down = HSI_dataset['down_votes'].iloc[(i-window)+1 : i+1]               #In the case that we do not have data values for x_(t-2), x_(t-1) and sometimes even x, we can treat it the upvotes and downvotes as 50-50, though this is not the best method, a better method would be to look up the news of the day and do a qualitiative approach or increase the time window, but that may make the regression locally inconsistent with noisy data
        open_i = HSI_dataset.loc[i - window + 1, 'close']
        close_i = HSI_dataset.loc[i, 'close']
        if (window_up.isna().sum() > (window//2)) or (window_down.isna().sum() > window//2):
            continue
        rolling_med_up = window_up.rolling(window, 2).median()                         # Calculating M_t for upvotes and downvotes 
        rolling_med_down = window_down.rolling(window, 2).median()                                                                                   
        devs_up = np.abs(window_up - rolling_med_up)                                   #Getting our individual data points to consider for MAD_t
        devs_down = np.abs(window_down - rolling_med_down)
        window_opens.append(open_i)
        window_closes.append(close_i)
        window_returns.append((close_i - open_i) / open_i)
        MAD_up[i] = devs_up.median()
        MAD_down[i] = devs_down.median()
        HSI_dataset.loc[i, 'close_percent_change_window'] = ((close_i - open_i) / open_i) * 100
    
    HSI_dataset['MAD_up'] = MAD_up                                                     #Keeping the Dataframe upto date, of course I can do this out of the loop and it would definitely be faster computationally but I am going to leave that a 'room for improvement' area on purpose.
    HSI_dataset['MAD_down'] = MAD_down
    HSI_dataset['window_open'] = np.nan
    HSI_dataset['window_close'] = np.nan
    
    HSI_dataset.loc[window - 1:, 'window_open'] = window_opens
    HSI_dataset.loc[window - 1:, 'window_close'] = window_closes
    HSI_dataset.loc[window - 1:, 'window_return'] = window_returns
     
    for i in range(len(HSI_dataset)):                                                  # Storing the upvote imputation values in an array to be appended into the excel file later
        if pd.isna(HSI_dataset.loc[i, 'up_votes']):                   
            MAD_candidates = []
            for j in [i, i-1, i-2]:
                if (0 <= j < len(HSI_dataset)) and not (pd.isna(HSI_dataset.loc[j, 'MAD_up'])) and (HSI_dataset.loc[j, 'MAD_up']) > 0:
                    MAD_candidates.append(HSI_dataset.loc[j, 'MAD_up'])
            if (min(MAD_candidates)) < MAD_threshold:                                                           #checking to see if the local properties of the time window are under the threshold that allows for interpolation
                HSI_dataset.loc[i, 'up_votes'] = HSI_dataset['up_votes'].interpolate(method='linear').iloc[i]   #using geometric linear interpolation istead of regression which would require fitting the data into a specific statistical distribution to find out the variance, std and error of the OHLC data and correlate somehow to percentage up vote and down vote data, which does not translate well. Also this is done within the loop over instead of being out of the loop as it allows us the ability to calculate the interpolated series for various time windows, giving us some manueverability if we want to change the window
            else:
                HSI_dataset.loc[i, 'up_votes'] = 50
    
        if pd.isna(HSI_dataset.loc[i, 'down_votes']):                                                           #down vote data time
            MAD_candidates = []
            for j in [i, i-1, i-2]:
                if (0 <= j < len(HSI_dataset)) and not (pd.isna(HSI_dataset.loc[j, 'MAD_down'])) and (HSI_dataset.loc[j, 'MAD_down']) > 0:
                    MAD_candidates.append(HSI_dataset.loc[j, 'MAD_down'])
            if (min(MAD_candidates)) < MAD_threshold:
                HSI_dataset.loc[i, 'down_votes'] = HSI_dataset['down_votes'].interpolate(method='linear').iloc[i]  #forgot to mention earlier but we use iloc because we do not want to run into numbering issues for indexes between the excel sheet and the pandas statistics here. For example if we take F2 cell in the excel file its index should be considered 1 within pandas even though it is the second cell in the given field.
            else:
                HSI_dataset.loc[i, 'down_votes'] = 50                                                              # 50 is a deliberate Bayesian prior for missing sentiment, not a data-free guess.
    
    HSI_dataset['net_votes'] = (HSI_dataset['up_votes'] - HSI_dataset['down_votes'])                          #getting the percentage value of total sentiment +ve percentage = bullish, -ve percentage = bearish
    #Finally we plot a scatter plot that compares the sentiment % change coded by the 'net_votes' field to the total percent change observed in every window on the HSI value and see if sentiment lines up with the actual market data, using mathplotlib
    fig, axis = plt.subplots(figsize=(12, 6))
    axis.plot(HSI_dataset['date'], HSI_dataset['net_votes'], label='Net Votes', color='blue')
    axis.plot(HSI_dataset['date'], HSI_dataset['close_percent_change_window'], label='Market Close in window', color='green')
    axis.set_xlabel('Date')
    axis.set_ylabel('%Change')
    axis.legend()
    plt.tight_layout()
    plt.show()

    HSI_dataset.to_csv("HSI_output.csv", index=False)                                                    #And lastly we create a csv based on our dataframe using pandas in built functions so we have a csv and a plot in the same current working directory for the project
    #return HSI_dataset                                                                                  #this is here in case you want to modify this into a function later for further data manipulation on the dataframe or to pass it along to MySQL and treat it as part of a larger schema 

    
HSI_Analyze(window=3, MAD_threshold=5)                                                       #call the procedure and we get all the data we need. Just need to ensure that all library files like mathplotlib, pandas and numpy are included.

ModuleNotFoundError: No module named 'matplotlib'

In [5]:
#In this bit of code I try to analyse the given spreadsheet here by measuring the market sentiment through 'up_votes' and 'down_votes' and see if there is any correlation between price movement and the sentiment.

#It is useful to note the role that time windows play in this code, as it serves as a means to control the analysis on local price action within a given time range, specifically by allowing us to choose the appropriate closing values of intra-time window periods instead of intra-day periods

#To keep it simple, the script does the following:
#- Reads the `HSI.xlsx` into a pandas DataFrame.
#- Parses the all data into appropriate data types (had to grant a special focus to date time as I was unsure if dates stored in a bold formate actually affect Pandas inbuilt fucntions, I then made the script remove all spaces in headers, because that can be a pain in dataframes while also making a copy of all the important original data for comparison purposes once the script outputs a csv file later
#- The script then uses rolling medians to calculate Median Absolute Deviation (I think I made a typo in the previous bin in this .ipynb file and called it absolute median deviation) to decide the regime of data imputation in case imputation is required upon parsing the data from out excel file
#- The imputation regime is then decided upon by the MAD value and a cutoff, if the MAD > cutoff then the missing data for sentiment (this is all to impute sentiment) is treated as 50% bullish, 50% bearish, otherwise we do a little geometric interpolation.
#- However, it is also possible to do some modelling of the overall dataset at first to allow for linear regression. I was just unable to find a good statistical model to fit the data onto. Regardless the sentiment we impute is stored as net_votes you can see how it works in the previous bin within this .ipynb
#- Lastly the code tries to focus on choosing the correct closure value within the given time window of local analysis (which is 3 days) and then converts it to %Change within given window
#- This allows me to use mathplotlib to compare the %Change and bullish/bearish sentiment in a scatter plot, pretty straight forward stuff

#If all works well you should get 2 outputs:
#- `HSI_output.csv`
#- a line plot comparing `net_votes` and window-based market return

#- And here are my assumptions: The analysis assumes the rows are already sorted correctly in time order.
#- The window operates on row position, not on calendar spacing.
#- Missing vote values are filled using a local MAD-based rule.
#- This is intended as an exploratory analysis workflow, not a production forecasting system.

SyntaxError: invalid syntax (947063450.py, line 3)